<a href="https://colab.research.google.com/github/eruskkii/unsupervised-learning-model-ecommerce/blob/main/Ecommerce_Unsupervised_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install KaggleHub, which lets us download public Kaggle datasets
!pip install -q kagglehub

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
import kagglehub

import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")


In [ ]:
# Download the Brazilian E-Commerce Public Dataset by Olist
dataset_path = kagglehub.dataset_download(
    "olistbr/brazilian-ecommerce"
)

dataset_path = Path(dataset_path)

print("Dataset downloaded to:")
print(dataset_path)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset downloaded to:
/kaggle/input/brazilian-ecommerce


In [ ]:
# Find every CSV file in the downloaded dataset folder
csv_files = sorted(dataset_path.glob("*.csv"))

print(f"Number of CSV files found: {len(csv_files)}\n")

for file_path in csv_files:
    print(file_path.name)

Number of CSV files found: 9

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [ ]:
# Map short, readable names to the corresponding CSV filenames
file_map = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

# Load every CSV file into a pandas DataFrame
data = {
    table_name: pd.read_csv(dataset_path / filename)
    for table_name, filename in file_map.items()
}

print(f"{len(data)} datasets loaded successfully.")

9 datasets loaded successfully.


In [ ]:
inventory_records = []

for table_name, df in data.items():
    inventory_records.append({
        "dataset": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum(),
        "memory_mb": df.memory_usage(deep=True).sum() / (1024 ** 2)
    })

inventory = (
    pd.DataFrame(inventory_records)
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

inventory

,dataset,rows,columns,duplicate_rows,missing_values,memory_mb
0,geolocation,1000163,5,261831,0,130.26
1,order_items,112650,7,0,0,35.99
2,payments,103886,5,0,0,16.23
3,customers,99441,5,0,0,26.59
4,orders,99441,8,0,4908,52.94
5,reviews,99224,7,0,145903,39.12
6,products,32951,9,0,2448,6.30
7,sellers,3095,4,0,0,0.59
8,category_translation,71,2,0,0,0.01


In [ ]:
def create_column_profile(df):
    """
    Create a column-level summary for a pandas DataFrame.
    """

    profile = pd.DataFrame({
        "column": df.columns,
        "data_type": df.dtypes.astype(str).values,
        "non_null_count": df.notna().sum().values,
        "missing_count": df.isna().sum().values,
        "missing_percentage": (
            df.isna().mean().mul(100).round(2).values
        ),
        "unique_values": df.nunique(dropna=True).values
    })

    return profile

In [ ]:
column_profiles = {}

for table_name, df in data.items():
    column_profiles[table_name] = create_column_profile(df)

print("Column profiles created successfully.")

Column profiles created successfully.


In [ ]:
for table_name, profile in column_profiles.items():
    print(f"\n{'=' * 70}")
    print(f"DATASET: {table_name.upper()}")
    print(f"{'=' * 70}")

    display(profile)


DATASET: CUSTOMERS


,column,data_type,non_null_count,missing_count,missing_percentage,unique_values
0,customer_id,object,99441,0,0.00,99441
1,customer_unique_id,object,99441,0,0.00,96096
2,customer_zip_code_prefix,int64,99441,0,0.00,14994
3,customer_city,object,99441,0,0.00,4119
4,customer_state,object,99441,0,0.00,27



DATASET: GEOLOCATION


,column,data_type,non_null_count,missing_count,missing_percentage,unique_values
0,geolocation_zip_code_prefix,int64,1000163,0,0.00,19015
1,geolocation_lat,float64,1000163,0,0.00,717360
2,geolocation_lng,float64,1000163,0,0.00,717613
3,geolocation_city,object,1000163,0,0.00,8011
4,geolocation_state,object,1000163,0,0.00,27



DATASET: ORDER_ITEMS


...

,column,data_type,non_null_count,missing_count,missing_percentage,unique_values
0,product_id,object,32951,0,0.00,32951
1,product_category_name,object,32341,610,1.85,73
2,product_name_lenght,float64,32341,610,1.85,66
3,product_description_lenght,float64,32341,610,1.85,2960
4,product_photos_qty,float64,32341,610,1.85,19
5,product_weight_g,float64,32949,2,0.01,2204
6,product_length_cm,float64,32949,2,0.01,99
7,product_height_cm,float64,32949,2,0.01,102
8,product_width_cm,float64,32949,2,0.01,95



DATASET: SELLERS


,column,data_type,non_null_count,missing_count,missing_percentage,unique_values
0,seller_id,object,3095,0,0.00,3095
1,seller_zip_code_prefix,int64,3095,0,0.00,2246
2,seller_city,object,3095,0,0.00,611
3,seller_state,object,3095,0,0.00,23



DATASET: CATEGORY_TRANSLATION


,column,data_type,non_null_count,missing_count,missing_percentage,unique_values
0,product_category_name,object,71,0,0.00,71
1,product_category_name_english,object,71,0,0.00,71


In [ ]:
for table_name, df in data.items():
    print(f"\n{'=' * 70}")
    print(f"DATASET: {table_name.upper()}")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"{'=' * 70}")

    display(df.head(3))


DATASET: CUSTOMERS
Shape: 99,441 rows × 5 columns


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



DATASET: GEOLOCATION
Shape: 1,000,163 rows × 5 columns


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.55,-46.64,sao paulo,SP
1,1046,-23.55,-46.64,sao paulo,SP
2,1046,-23.55,-46.64,sao paulo,SP



DATASET: ORDER_ITEMS
Shape: 112,650 rows × 7 columns


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,"1,000.00",30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00



DATASET: SELLERS
Shape: 3,095 rows × 4 columns


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ



DATASET: CATEGORY_TRANSLATION
Shape: 71 rows × 2 columns


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [ ]:
missing_records = []

for table_name, df in data.items():
    missing_counts = df.isna().sum()

    for column_name, missing_count in missing_counts.items():
        if missing_count > 0:
            missing_records.append({
                "dataset": table_name,
                "column": column_name,
                "missing_count": missing_count,
                "missing_percentage": round(
                    missing_count / len(df) * 100, 2
                )
            })

missing_summary = (
    pd.DataFrame(missing_records)
    .sort_values(
        ["dataset", "missing_percentage"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

missing_summary

,dataset,column,missing_count,missing_percentage
0,orders,order_delivered_customer_date,2965,2.98
1,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_approved_at,160,0.16
3,products,product_category_name,610,1.85
4,products,product_name_lenght,610,1.85
5,products,product_description_lenght,610,1.85
6,products,product_photos_qty,610,1.85
7,products,product_weight_g,2,0.01
8,products,product_length_cm,2,0.01
9,products,product_height_cm,2,0.01


### Initial data-quality observations

The nine datasets were successfully loaded and inspected. The geolocation
dataset is the largest table, containing 1,000,163 records and 261,831 exact
duplicate rows. These records will be investigated further because repeated
postcode prefixes may be legitimate, whereas completely identical records
may be redundant.

The customers and orders datasets each contain 99,441 records. The
`customer_id` column is unique in both tables, indicating a one-to-one
relationship between an order and its order-specific customer record.
However, the customers table contains only 96,096 distinct
`customer_unique_id` values. This indicates that some underlying customers
placed more than one order. Consequently, `customer_unique_id`, rather than
`customer_id`, will be used when constructing customer-level behavioural
features.

Missing values are concentrated in the orders, products and reviews
datasets. The missing delivery-related dates in the orders table may reflect
cancelled, unavailable, shipped or otherwise undelivered orders and will be
analysed against order status before any treatment is applied. Review titles
and comments are optional text fields, so their absence will not be treated
as evidence of an invalid review. Missing product metadata will also be
investigated to determine whether the affected fields are absent from the
same product records.

No values have been deleted or imputed at this stage. Cleaning decisions will
be made only after investigating the meaning and analytical consequences of
each issue.

In [ ]:
customers = data["customers"]
orders = data["orders"]

key_checks = pd.DataFrame({
    "check": [
        "customers.customer_id is unique",
        "orders.order_id is unique",
        "orders.customer_id is unique",
        "Every order customer_id exists in customers",
        "Every customer customer_id exists in orders"
    ],
    "result": [
        customers["customer_id"].is_unique,
        orders["order_id"].is_unique,
        orders["customer_id"].is_unique,
        orders["customer_id"].isin(customers["customer_id"]).all(),
        customers["customer_id"].isin(orders["customer_id"]).all()
    ]
})

key_checks

,check,result
0,customers.customer_id is unique,True
1,orders.order_id is unique,True
2,orders.customer_id is unique,True
3,Every order customer_id exists in customers,True
4,Every customer customer_id exists in orders,True


In [ ]:
orders_with_customers = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print(f"Orders before merge: {len(orders):,}")
print(f"Rows after merge:     {len(orders_with_customers):,}")

orders_with_customers.head()

Orders before merge: 99,441
Rows after merge:     99,441


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [ ]:
customer_order_counts = (
    orders_with_customers
    .groupby("customer_unique_id")
    .size()
    .reset_index(name="number_of_orders")
)

repeat_customer_summary = pd.DataFrame({
    "metric": [
        "Unique customers",
        "One-time customers",
        "Repeat customers",
        "Maximum orders by one customer",
        "Repeat-customer percentage"
    ],
    "value": [
        customer_order_counts["customer_unique_id"].nunique(),
        (customer_order_counts["number_of_orders"] == 1).sum(),
        (customer_order_counts["number_of_orders"] > 1).sum(),
        customer_order_counts["number_of_orders"].max(),
        round(
            (customer_order_counts["number_of_orders"] > 1).mean() * 100,
            2
        )
    ]
})

repeat_customer_summary

,metric,value
0,Unique customers,"96,096.00"
1,One-time customers,"93,099.00"
2,Repeat customers,"2,997.00"
3,Maximum orders by one customer,17.00
4,Repeat-customer percentage,3.12


In [ ]:
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

In [ ]:
for column in order_date_columns:
    orders_with_customers[column] = pd.to_datetime(
        orders_with_customers[column],
        errors="coerce"
    )

orders_with_customers[order_date_columns].dtypes

,0
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


In [ ]:
date_conversion_check = pd.DataFrame({
    "column": order_date_columns,
    "data_type": [
        orders_with_customers[column].dtype
        for column in order_date_columns
    ],
    "missing_after_conversion": [
        orders_with_customers[column].isna().sum()
        for column in order_date_columns
    ]
})

date_conversion_check

,column,data_type,missing_after_conversion
0,order_purchase_timestamp,datetime64[ns],0
1,order_approved_at,datetime64[ns],160
2,order_delivered_carrier_date,datetime64[ns],1783
3,order_delivered_customer_date,datetime64[ns],2965
4,order_estimated_delivery_date,datetime64[ns],0


In [ ]:
order_status_quality = (
    orders_with_customers
    .assign(
        approval_date_missing=
            orders_with_customers["order_approved_at"].isna(),

        carrier_date_missing=
            orders_with_customers["order_delivered_carrier_date"].isna(),

        customer_delivery_date_missing=
            orders_with_customers["order_delivered_customer_date"].isna()
    )
    .groupby("order_status")
    .agg(
        number_of_orders=("order_id", "count"),
        missing_approval_dates=("approval_date_missing", "sum"),
        missing_carrier_dates=("carrier_date_missing", "sum"),
        missing_customer_delivery_dates=(
            "customer_delivery_date_missing", "sum"
        )
    )
    .sort_values("number_of_orders", ascending=False)
    .reset_index()
)

order_status_quality

,order_status,number_of_orders,missing_approval_dates,missing_carrier_dates,missing_customer_delivery_dates
0,delivered,96478,14,2,8
1,shipped,1107,0,0,1107
2,canceled,625,141,550,619
3,unavailable,609,0,609,609
4,invoiced,314,0,314,314
5,processing,301,0,301,301
6,created,5,5,5,5
7,approved,2,0,2,2


### Customer identity and order-status findings

Key validation confirmed that `customer_id` is unique in both the customers
and orders datasets. Every order has a corresponding customer record and
every customer record is associated with an order. The two tables therefore
have a verified one-to-one relationship at the order-specific `customer_id`
level.

The 99,441 order-specific customer records correspond to 96,096 underlying
customers. Of these customers, 93,099 placed only one order, while 2,997
placed more than one order. Repeat purchasers therefore represent only 3.12%
of identified customers. The highest number of orders associated with one
customer is 17. This low repeat-purchase rate is potentially commercially
significant, although it cannot by itself be interpreted as proof of poor
retention. Product-replacement cycles, identity resolution, the observation
period and marketplace purchasing behaviour may also influence the result.

Conversion of the five order-date columns to `datetime` introduced no
additional missing values. Most missing operational dates are consistent
with order status. Shipped orders lack customer-delivery dates, while
unavailable, invoiced, processing and created orders had not reached the
corresponding fulfilment stages.

A small number of inconsistencies remain. Some orders marked as delivered
lack approval, carrier or customer-delivery dates. In addition, six cancelled
orders contain a customer-delivery date. These records will be inspected
individually and excluded only from analyses requiring the affected dates.
They will not be deleted from the complete dataset without further evidence.

In [ ]:
delivered_date_anomalies = orders_with_customers.loc[
    (orders_with_customers["order_status"] == "delivered")
    &
    (
        orders_with_customers["order_approved_at"].isna()
        | orders_with_customers["order_delivered_carrier_date"].isna()
        | orders_with_customers["order_delivered_customer_date"].isna()
    ),
    [
        "order_id",
        "customer_unique_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].sort_values("order_purchase_timestamp")

print(
    f"Delivered orders with at least one missing operational date: "
    f"{len(delivered_date_anomalies)}"
)

display(delivered_date_anomalies)

Delivered orders with at least one missing operational date: 23


,order_id,customer_unique_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
26800,c1d4211b3dae76144deccd6c74144a88,6ff8b0d7b35d5c945633b8d60165691b,delivered,2017-01-19 12:48:08,NaT,2017-01-25 14:56:50,2017-01-30 18:16:01,2017-03-01
48401,7002a78c79c519ac54022d4f8a65e6e8,d49f3dae6bad25d05160fc17aca5942d,delivered,2017-01-19 22:26:59,NaT,2017-01-27 11:08:05,2017-02-06 14:22:19,2017-03-16
23156,12a95a3c06dbaec84bcfb0e2da5d228a,c8822fce1d0bfa7ddf0da24fff947172,delivered,2017-02-17 13:05:55,NaT,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20
72407,3c0b8706b065f9919d0505d3b3343881,c094ac95fcd52f821809ec232a7a6956,delivered,2017-02-17 15:53:27,NaT,2017-02-22 11:31:30,2017-03-03 11:47:47,2017-03-23
61743,2eecb0d85f281280f79fa00f9cec1a95,5a4fa4919cbf2b049e72be460a380e5b,delivered,2017-02-17 17:21:55,NaT,2017-02-22 11:42:51,2017-03-03 12:16:03,2017-03-20
39334,d77031d6a3c8a52f019764e68f211c69,c4c0011e639bdbcf26059ddc38bd3c18,delivered,2017-02-18 11:04:19,NaT,2017-02-23 07:23:36,2017-03-02 16:15:23,2017-03-22
16567,8a9adc69528e1001fc68dd0aaebbb54a,91efb7fcabc17925099dced52435837f,delivered,2017-02-18 12:45:31,NaT,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21
19031,7013bcfc1c97fe719a7b5e05e61c12db,e1f01a1bd6485e58ad3c769a5427d8a8,delivered,2017-02-18 13:29:47,NaT,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17
5323,e04abd8149ef81b95221e88f6ed9ab6a,8a9a08c7ca8900a200d83cf838a07e0b,delivered,2017-02-18 14:40:00,NaT,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17
63052,51eb2eebd5d76a24625b31c33dd41449,79af1bbf230a2630487975aa5d7d6220,delivered,2017-02-18 15:52:27,NaT,2017-02-23 03:09:14,2017-03-07 13:57:47,2017-03-29


In [ ]:
non_delivered_with_delivery_date = orders_with_customers.loc[
    (orders_with_customers["order_status"] != "delivered")
    & orders_with_customers["order_delivered_customer_date"].notna(),
    [
        "order_id",
        "customer_unique_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].sort_values("order_purchase_timestamp")

print(
    f"Non-delivered orders with a recorded customer-delivery date: "
    f"{len(non_delivered_with_delivery_date)}"
)

display(non_delivered_with_delivery_date)

Non-delivered orders with a recorded customer-delivery date: 6


,order_id,customer_unique_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
92636,65d1e226dfaeb8cdc42f665422522d14,b8b8726af116a5cfb35b0315ecef9172,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25
58266,770d331c84e5b214bd9dc70a10b829d0,59a7bb642795ddc85534fbd5f7360334,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29
59332,8beb59392e21af5eb9547ae1a9938d06,f9af3bfab584769f34c314b222d34263,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30
8791,dabf2b0e35b423f94618bf965fcb7514,2f96790fba243100730c8d1185dd2f25,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,d93a22bb52bb2f41a7267b8a02370781,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08
2921,1950d777989f6a877539f53795b4c3c3,6a04c8339c44b23e189446216a494569,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09


In [ ]:
candidate_keys = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "reviews_by_review_id": ["review_id"],
    "reviews_by_order_id": ["order_id"],
    "category_translation": ["product_category_name"]
}

candidate_key_results = []

for key_name, key_columns in candidate_keys.items():

    if key_name == "reviews_by_review_id":
        table_name = "reviews"
    elif key_name == "reviews_by_order_id":
        table_name = "reviews"
    else:
        table_name = key_name

    df = data[table_name]

    duplicate_count = df.duplicated(
        subset=key_columns,
        keep=False
    ).sum()

    candidate_key_results.append({
        "candidate_key": key_name,
        "columns": ", ".join(key_columns),
        "row_count": len(df),
        "rows_in_duplicate_groups": duplicate_count,
        "is_unique": duplicate_count == 0
    })

candidate_key_summary = pd.DataFrame(candidate_key_results)

candidate_key_summary

,candidate_key,columns,row_count,rows_in_duplicate_groups,is_unique
0,customers,customer_id,99441,0,True
1,orders,order_id,99441,0,True
2,order_items,"order_id, order_item_id",112650,0,True
3,payments,"order_id, payment_sequential",103886,0,True
4,products,product_id,32951,0,True
5,sellers,seller_id,3095,0,True
6,reviews_by_review_id,review_id,99224,1603,False
7,reviews_by_order_id,order_id,99224,1098,False
8,category_translation,product_category_name,71,0,True


In [ ]:
def check_foreign_key(
    child_df,
    child_column,
    parent_df,
    parent_column,
    relationship_name
):
    """
    Check whether non-missing values in a child-table column
    exist in the corresponding parent-table column.
    """

    child_values = set(
        child_df[child_column].dropna().unique()
    )

    parent_values = set(
        parent_df[parent_column].dropna().unique()
    )

    unmatched_values = child_values - parent_values

    return {
        "relationship": relationship_name,
        "distinct_child_keys": len(child_values),
        "unmatched_keys": len(unmatched_values),
        "all_keys_matched": len(unmatched_values) == 0
    }

In [ ]:
foreign_key_results = [
    check_foreign_key(
        data["orders"],
        "customer_id",
        data["customers"],
        "customer_id",
        "orders.customer_id → customers.customer_id"
    ),

    check_foreign_key(
        data["order_items"],
        "order_id",
        data["orders"],
        "order_id",
        "order_items.order_id → orders.order_id"
    ),

    check_foreign_key(
        data["order_items"],
        "product_id",
        data["products"],
        "product_id",
        "order_items.product_id → products.product_id"
    ),

    check_foreign_key(
        data["order_items"],
        "seller_id",
        data["sellers"],
        "seller_id",
        "order_items.seller_id → sellers.seller_id"
    ),

    check_foreign_key(
        data["payments"],
        "order_id",
        data["orders"],
        "order_id",
        "payments.order_id → orders.order_id"
    ),

    check_foreign_key(
        data["reviews"],
        "order_id",
        data["orders"],
        "order_id",
        "reviews.order_id → orders.order_id"
    ),

    check_foreign_key(
        data["products"],
        "product_category_name",
        data["category_translation"],
        "product_category_name",
        "products.category → category translation"
    ),

    check_foreign_key(
        data["customers"],
        "customer_zip_code_prefix",
        data["geolocation"],
        "geolocation_zip_code_prefix",
        "customers postcode → geolocation postcode"
    ),

    check_foreign_key(
        data["sellers"],
        "seller_zip_code_prefix",
        data["geolocation"],
        "geolocation_zip_code_prefix",
        "sellers postcode → geolocation postcode"
    )
]

foreign_key_summary = pd.DataFrame(foreign_key_results)

foreign_key_summary

,relationship,distinct_child_keys,unmatched_keys,all_keys_matched
0,orders.customer_id → customers.customer_id,99441,0,True
1,order_items.order_id → orders.order_id,98666,0,True
2,order_items.product_id → products.product_id,32951,0,True
3,order_items.seller_id → sellers.seller_id,3095,0,True
4,payments.order_id → orders.order_id,99440,0,True
5,reviews.order_id → orders.order_id,98673,0,True
6,products.category → category translation,73,2,False
7,customers postcode → geolocation postcode,14994,157,False
8,sellers postcode → geolocation postcode,2246,7,False


### Dataset-key and relationship assessment

Candidate-key testing confirmed that the customers, orders, products and
sellers datasets have unique single-column identifiers. Order items are
uniquely identified by the composite key of `order_id` and `order_item_id`,
while payments are uniquely identified by `order_id` and
`payment_sequential`.

The reviews dataset does not have a simple one-to-one relationship with
orders. Both `review_id` and `order_id` occur in duplicate groups. Reviews
will therefore be investigated and converted to an order-level summary before
being joined to the orders dataset.

All order, product, seller, payment and review foreign keys refer to valid
parent records. Nevertheless, some orders have no corresponding order-item,
payment or review records. The status distribution of these orders will be
examined before determining whether their absence is expected.

Two non-missing Portuguese product categories do not have corresponding
English translations. In addition, 157 customer postcode prefixes and seven
seller postcode prefixes are absent from the geolocation dataset. Geographic
and translation data will consequently be attached using left joins so that
unmatched transactional records are retained.

Operational-date anomalies will not be globally deleted or imputed. Records
will be excluded only from analyses that require a missing or contradictory
timestamp. Completed-purchase analyses will use orders whose recorded status
is `delivered`.

In [ ]:
product_categories = set(
    data["products"]["product_category_name"]
    .dropna()
    .unique()
)

translated_categories = set(
    data["category_translation"]["product_category_name"]
    .dropna()
    .unique()
)

untranslated_categories = sorted(
    product_categories - translated_categories
)

print("Categories without English translations:")
print(untranslated_categories)

Categories without English translations:
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


In [ ]:
untranslated_category_counts = (
    data["products"]
    .loc[
        data["products"]["product_category_name"]
        .isin(untranslated_categories),
        "product_category_name"
    ]
    .value_counts()
    .rename_axis("product_category_name")
    .reset_index(name="number_of_products")
)

untranslated_category_counts

,product_category_name,number_of_products
0,portateis_cozinha_e_preparadores_de_alimentos,10
1,pc_gamer,3


In [ ]:
geolocation_postcodes = set(
    data["geolocation"]["geolocation_zip_code_prefix"]
    .dropna()
    .unique()
)

unmatched_customer_postcodes = sorted(
    set(data["customers"]["customer_zip_code_prefix"].unique())
    - geolocation_postcodes
)

unmatched_seller_postcodes = sorted(
    set(data["sellers"]["seller_zip_code_prefix"].unique())
    - geolocation_postcodes
)

postcode_mismatch_summary = pd.DataFrame({
    "entity": ["customers", "sellers"],
    "unmatched_distinct_postcodes": [
        len(unmatched_customer_postcodes),
        len(unmatched_seller_postcodes)
    ],
    "affected_records": [
        data["customers"]["customer_zip_code_prefix"]
        .isin(unmatched_customer_postcodes)
        .sum(),

        data["sellers"]["seller_zip_code_prefix"]
        .isin(unmatched_seller_postcodes)
        .sum()
    ],
    "percentage_of_records": [
        round(
            data["customers"]["customer_zip_code_prefix"]
            .isin(unmatched_customer_postcodes)
            .mean() * 100,
            2
        ),

        round(
            data["sellers"]["seller_zip_code_prefix"]
            .isin(unmatched_seller_postcodes)
            .mean() * 100,
            2
        )
    ]
})

postcode_mismatch_summary

,entity,unmatched_distinct_postcodes,affected_records,percentage_of_records
0,customers,157,278,0.28
1,sellers,7,7,0.23


In [ ]:
order_coverage = orders[
    ["order_id", "order_status"]
].copy()

order_item_ids = set(
    data["order_items"]["order_id"].unique()
)

payment_order_ids = set(
    data["payments"]["order_id"].unique()
)

review_order_ids = set(
    data["reviews"]["order_id"].unique()
)

order_coverage["has_order_items"] = (
    order_coverage["order_id"].isin(order_item_ids)
)

order_coverage["has_payment"] = (
    order_coverage["order_id"].isin(payment_order_ids)
)

order_coverage["has_review"] = (
    order_coverage["order_id"].isin(review_order_ids)
)

order_coverage.head()

,order_id,order_status,has_order_items,has_payment,has_review
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,True,True,True
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,True,True,True
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,True,True,True
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,True,True,True
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,True,True,True


In [ ]:
coverage_by_status = (
    order_coverage
    .assign(
        missing_order_items=~order_coverage["has_order_items"],
        missing_payment=~order_coverage["has_payment"],
        missing_review=~order_coverage["has_review"]
    )
    .groupby("order_status")
    .agg(
        number_of_orders=("order_id", "count"),
        orders_without_items=("missing_order_items", "sum"),
        orders_without_payment=("missing_payment", "sum"),
        orders_without_review=("missing_review", "sum")
    )
    .sort_values("number_of_orders", ascending=False)
    .reset_index()
)

coverage_by_status

,order_status,number_of_orders,orders_without_items,orders_without_payment,orders_without_review
0,delivered,96478,0,1,646
1,shipped,1107,1,0,75
2,canceled,625,164,0,20
3,unavailable,609,603,0,14
4,invoiced,314,2,0,5
5,processing,301,0,0,6
6,created,5,5,0,2
7,approved,2,0,0,0


In [ ]:
reviews = data["reviews"].copy()

review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in review_date_columns:
    reviews[column] = pd.to_datetime(
        reviews[column],
        errors="coerce"
    )

reviews[review_date_columns].dtypes

,0
review_creation_date,datetime64[ns]
review_answer_timestamp,datetime64[ns]


In [ ]:
review_id_groups = (
    reviews
    .groupby("review_id")
    .agg(
        number_of_rows=("review_id", "size"),
        distinct_orders=("order_id", "nunique"),
        distinct_scores=("review_score", "nunique"),
        earliest_answer=("review_answer_timestamp", "min"),
        latest_answer=("review_answer_timestamp", "max")
    )
    .reset_index()
)

order_review_groups = (
    reviews
    .groupby("order_id")
    .agg(
        number_of_review_rows=("order_id", "size"),
        distinct_review_ids=("review_id", "nunique"),
        distinct_scores=("review_score", "nunique"),
        earliest_answer=("review_answer_timestamp", "min"),
        latest_answer=("review_answer_timestamp", "max")
    )
    .reset_index()
)

In [ ]:
review_duplicate_structure = pd.DataFrame({
    "metric": [
        "Distinct review IDs",
        "Review IDs appearing in multiple rows",
        "Review IDs linked to multiple orders",
        "Distinct reviewed orders",
        "Orders with multiple review rows",
        "Orders with multiple review IDs",
        "Orders with conflicting review scores"
    ],
    "value": [
        reviews["review_id"].nunique(),

        (review_id_groups["number_of_rows"] > 1).sum(),

        (review_id_groups["distinct_orders"] > 1).sum(),

        reviews["order_id"].nunique(),

        (order_review_groups["number_of_review_rows"] > 1).sum(),

        (order_review_groups["distinct_review_ids"] > 1).sum(),

        (order_review_groups["distinct_scores"] > 1).sum()
    ]
})

review_duplicate_structure

,metric,value
0,Distinct review IDs,98410
1,Review IDs appearing in multiple rows,789
2,Review IDs linked to multiple orders,789
3,Distinct reviewed orders,98673
4,Orders with multiple review rows,547
5,Orders with multiple review IDs,547
6,Orders with conflicting review scores,202


In [ ]:
multi_review_order_ids = (
    order_review_groups
    .loc[
        order_review_groups["number_of_review_rows"] > 1,
        "order_id"
    ]
)

multi_review_sample = (
    reviews[
        reviews["order_id"].isin(
            multi_review_order_ids.head(10)
        )
    ]
    .sort_values(
        ["order_id", "review_answer_timestamp"]
    )
)

display(
    multi_review_sample[
        [
            "order_id",
            "review_id",
            "review_score",
            "review_comment_title",
            "review_comment_message",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    ]
)

,order_id,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
22423,0035246a40f520710769010f752e7507,2a74b0559eb58fc1ff842ecc999594cb,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25,2017-08-29 21:45:57
25612,0035246a40f520710769010f752e7507,89a02c45c340aeeb1354a24e7d4b2c1e,5,NaN,NaN,2017-08-29,2017-08-30 01:59:12
22779,013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,NaN,NaN,2018-02-22,2018-02-23 12:12:30
68633,013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,NaN,NaN,2018-03-04,2018-03-05 17:02:00
854,0176a6846bcb3b0d3aa3116a9a768597,830636803620cdf8b6ffaf1b2f6e92b2,5,NaN,NaN,2017-12-30,2018-01-02 10:54:06
83224,0176a6846bcb3b0d3aa3116a9a768597,d8e8c42271c8fb67b9dad95d98c8ff80,5,NaN,NaN,2017-12-30,2018-01-02 10:54:47
89888,02355020fd0a40a0d56df9f6ff060413,0c8e7347f1cdd2aede37371543e3d163,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21,2018-03-22 01:32:08
17582,02355020fd0a40a0d56df9f6ff060413,017f0e1ea6386de662cbeba299c59ad1,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29,2018-03-30 03:16:19
37911,029863af4b968de1e5d6a82782e662f5,04d945e95c788a3aa1ffbee42105637b,5,NaN,NaN,2017-07-14,2017-07-17 13:58:06
55137,029863af4b968de1e5d6a82782e662f5,61fe4e7d1ae801bbe169eb67b86c6eda,4,NaN,NaN,2017-07-19,2017-07-20 12:06:11


In [ ]:
shared_review_ids = review_id_groups.loc[
    review_id_groups["distinct_orders"] > 1,
    "review_id"
]

shared_review_id_sample = (
    reviews[
        reviews["review_id"].isin(
            shared_review_ids.head(10)
        )
    ]
    .sort_values(
        ["review_id", "review_answer_timestamp"]
    )
)

display(
    shared_review_id_sample[
        [
            "review_id",
            "order_id",
            "review_score",
            "review_comment_title",
            "review_comment_message",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    ]
)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44


### Cleaning decisions arising from relationship analysis

The review dataset contains two forms of duplication. Some `review_id` values
are attached to multiple orders with identical scores, text and timestamps.
These associations will be retained because the orders are distinct.
Separately, 547 orders contain multiple review records, and 202 of these
orders contain conflicting scores. These records may represent updated
customer opinions. Reviews will therefore be sorted chronologically and
summarised at order level. The most recent review will represent the final
recorded opinion, while review count, mean score, score range and score-change
indicators will preserve information about the review history.

Missing order-item records are concentrated among unavailable and cancelled
orders. No delivered order lacks an order-item record. One delivered order
lacks a payment record and will be investigated before its monetary value is
treated. Delivered orders without reviews will be retained because customer
reviews are optional.

Thirteen products belong to two Portuguese categories absent from the
translation table. Analytical English labels will be added manually and
documented as project-created labels rather than official Olist translations.

Postcodes missing from the geolocation table affect 0.28% of customer records
and 0.23% of seller records. These records will not be deleted. They will
remain in non-geographic analyses and will be excluded only when valid
coordinates are required.

In [ ]:
manual_category_translations = pd.DataFrame({
    "product_category_name": [
        "portateis_cozinha_e_preparadores_de_alimentos",
        "pc_gamer"
    ],
    "product_category_name_english": [
        "portable_kitchen_and_food_preparation",
        "pc_gamer"
    ]
})

category_translation_clean = pd.concat(
    [
        data["category_translation"],
        manual_category_translations
    ],
    ignore_index=True
)

category_translation_clean.tail()

,product_category_name,product_category_name_english
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes
70,seguros_e_servicos,security_and_services
71,portateis_cozinha_e_preparadores_de_alimentos,portable_kitchen_and_food_preparation
72,pc_gamer,pc_gamer


In [ ]:
print(
    "Translation rows:",
    len(category_translation_clean)
)

print(
    "Unique Portuguese categories:",
    category_translation_clean[
        "product_category_name"
    ].nunique()
)

print(
    "Duplicated Portuguese categories:",
    category_translation_clean.duplicated(
        subset="product_category_name"
    ).sum()
)

Translation rows: 73
Unique Portuguese categories: 73
Duplicated Portuguese categories: 0


In [ ]:
products_clean = data["products"].merge(
    category_translation_clean,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

products_clean[
    "product_category_name_english"
] = products_clean[
    "product_category_name_english"
].fillna("unknown_category")

In [ ]:
product_cleaning_check = pd.DataFrame({
    "metric": [
        "Original product rows",
        "Clean product rows",
        "Missing English categories",
        "Unknown-category products"
    ],
    "value": [
        len(data["products"]),
        len(products_clean),
        products_clean[
            "product_category_name_english"
        ].isna().sum(),
        (
            products_clean[
                "product_category_name_english"
            ] == "unknown_category"
        ).sum()
    ]
})

product_cleaning_check

,metric,value
0,Original product rows,32951
1,Clean product rows,32951
2,Missing English categories,0
3,Unknown-category products,610


In [ ]:
reviews["has_review_title"] = (
    reviews["review_comment_title"]
    .notna()
    .astype(int)
)

reviews["has_review_comment"] = (
    reviews["review_comment_message"]
    .notna()
    .astype(int)
)

In [ ]:
review_history_summary = (
    reviews
    .groupby("order_id")
    .agg(
        review_record_count=("review_id", "size"),
        distinct_review_ids=("review_id", "nunique"),
        mean_review_score=("review_score", "mean"),
        minimum_review_score=("review_score", "min"),
        maximum_review_score=("review_score", "max"),
        any_review_title=("has_review_title", "max"),
        any_review_comment=("has_review_comment", "max")
    )
    .reset_index()
)

review_history_summary["review_score_changed"] = (
    review_history_summary["minimum_review_score"]
    != review_history_summary["maximum_review_score"]
).astype(int)

In [ ]:
latest_reviews = (
    reviews
    .sort_values(
        [
            "order_id",
            "review_answer_timestamp",
            "review_creation_date"
        ]
    )
    .drop_duplicates(
        subset="order_id",
        keep="last"
    )
    [
        [
            "order_id",
            "review_id",
            "review_score",
            "has_review_title",
            "has_review_comment",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    ]
    .rename(
        columns={
            "review_id": "latest_review_id",
            "review_score": "final_review_score",
            "has_review_title": "final_review_has_title",
            "has_review_comment": "final_review_has_comment",
            "review_creation_date": "final_review_creation_date",
            "review_answer_timestamp": "final_review_answer_timestamp"
        }
    )
)

In [ ]:
order_review_summary = review_history_summary.merge(
    latest_reviews,
    on="order_id",
    how="left",
    validate="one_to_one"
)

order_review_summary.head()

,order_id,review_record_count,distinct_review_ids,mean_review_score,minimum_review_score,maximum_review_score,any_review_title,any_review_comment,review_score_changed,latest_review_id,final_review_score,final_review_has_title,final_review_has_comment,final_review_creation_date,final_review_answer_timestamp
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,5.00,5,5,0,1,0,97ca439bc427b48bc1cd7177abe71365,5,0,1,2017-09-21,2017-09-22 10:57:03
1,00018f77f2f0320c557190d7a144bdd3,1,1,4.00,4,4,0,0,0,7b07bacd811c4117b742569b04ce3580,4,0,0,2017-05-13,2017-05-15 11:34:13
2,000229ec398224ef6ca0657da4fc703e,1,1,5.00,5,5,0,1,0,0c5b33dea94867d1ac402749e5438e8b,5,0,1,2018-01-23,2018-01-23 16:06:31
3,00024acbcdf0a6daa1e931b038114c75,1,1,4.00,4,4,0,0,0,f4028d019cb58564807486a6aaf33817,4,0,0,2018-08-15,2018-08-15 16:39:01
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,5.00,5,5,0,1,0,940144190dcba6351888cafa43f3a3a5,5,0,1,2017-03-02,2017-03-03 10:54:59


In [ ]:
review_summary_validation = pd.DataFrame({
    "metric": [
        "Distinct reviewed orders in original data",
        "Rows in order-level review summary",
        "Unique order IDs in review summary",
        "Orders with changed review scores",
        "Orders with more than one review record",
        "Missing final review scores"
    ],
    "value": [
        reviews["order_id"].nunique(),
        len(order_review_summary),
        order_review_summary["order_id"].nunique(),
        order_review_summary["review_score_changed"].sum(),
        (
            order_review_summary["review_record_count"] > 1
        ).sum(),
        order_review_summary["final_review_score"].isna().sum()
    ]
})

review_summary_validation

,metric,value
0,Distinct reviewed orders in original data,98673
1,Rows in order-level review summary,98673
2,Unique order IDs in review summary,98673
3,Orders with changed review scores,202
4,Orders with more than one review record,547
5,Missing final review scores,0


In [ ]:
geolocation_original = data["geolocation"]

geolocation_deduplicated = (
    geolocation_original
    .drop_duplicates()
    .copy()
)

geolocation_duplicate_check = pd.DataFrame({
    "metric": [
        "Original rows",
        "Rows after removing exact duplicates",
        "Exact duplicate rows removed"
    ],
    "value": [
        len(geolocation_original),
        len(geolocation_deduplicated),
        len(geolocation_original)
        - len(geolocation_deduplicated)
    ]
})

geolocation_duplicate_check

,metric,value
0,Original rows,1000163
1,Rows after removing exact duplicates,738332
2,Exact duplicate rows removed,261831


In [ ]:
geolocation_by_postcode = (
    geolocation_deduplicated
    .groupby("geolocation_zip_code_prefix")
    .agg(
        latitude=("geolocation_lat", "median"),
        longitude=("geolocation_lng", "median"),
        coordinate_observations=("geolocation_lat", "size"),
        distinct_cities=("geolocation_city", "nunique"),
        distinct_states=("geolocation_state", "nunique")
    )
    .reset_index()
)

geolocation_by_postcode.head()

,geolocation_zip_code_prefix,latitude,longitude,coordinate_observations,distinct_cities,distinct_states
0,1001,-23.55,-46.63,11,2,1
1,1002,-23.55,-46.64,6,2,1
2,1003,-23.55,-46.64,11,2,1
3,1004,-23.55,-46.63,14,2,1
4,1005,-23.55,-46.64,13,2,1


In [ ]:
geolocation_summary_validation = pd.DataFrame({
    "metric": [
        "Distinct postcode prefixes in source",
        "Rows in postcode summary",
        "Unique postcodes in summary"
    ],
    "value": [
        geolocation_deduplicated[
            "geolocation_zip_code_prefix"
        ].nunique(),
        len(geolocation_by_postcode),
        geolocation_by_postcode[
            "geolocation_zip_code_prefix"
        ].nunique()
    ]
})

geolocation_summary_validation

,metric,value
0,Distinct postcode prefixes in source,19015
1,Rows in postcode summary,19015
2,Unique postcodes in summary,19015


In [ ]:
order_items = data["order_items"].copy()

order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

order_items["item_total"] = (
    order_items["price"]
    + order_items["freight_value"]
)

order_item_summary = (
    order_items
    .groupby("order_id")
    .agg(
        item_record_count=("order_item_id", "size"),
        maximum_item_sequence=("order_item_id", "max"),
        distinct_products=("product_id", "nunique"),
        distinct_sellers=("seller_id", "nunique"),
        merchandise_value=("price", "sum"),
        freight_value=("freight_value", "sum"),
        item_based_total=("item_total", "sum")
    )
    .reset_index()
)

order_item_summary.head()

,order_id,item_record_count,maximum_item_sequence,distinct_products,distinct_sellers,merchandise_value,freight_value,item_based_total
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,1,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,1,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,1,1,1,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,1,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,1,199.90,18.14,218.04


In [ ]:
payments = data["payments"].copy()

payment_summary = (
    payments
    .groupby("order_id")
    .agg(
        payment_record_count=("payment_sequential", "size"),
        number_of_payment_types=("payment_type", "nunique"),
        maximum_installments=("payment_installments", "max"),
        payment_total=("payment_value", "sum")
    )
    .reset_index()
)

payment_summary.head()

,order_id,payment_record_count,number_of_payment_types,maximum_installments,payment_total
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,2,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,1,3,259.83
2,000229ec398224ef6ca0657da4fc703e,1,1,5,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,2,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,3,218.04


In [ ]:
delivered_without_payment = (
    orders_with_customers[
        (orders_with_customers["order_status"] == "delivered")
        & ~orders_with_customers["order_id"].isin(
            payment_summary["order_id"]
        )
    ]
)

display(
    delivered_without_payment[
        [
            "order_id",
            "customer_unique_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date"
        ]
    ]
)

,order_id,customer_unique_id,order_status,order_purchase_timestamp,order_delivered_customer_date
30710,bfbd0f9bdef84302105ad712db648a6c,830d5b7aaa3b6f1e9ad63703bec97d23,delivered,2016-09-15 12:16:38,2016-11-09 07:47:38


In [ ]:
delivered_without_payment_details = (
    delivered_without_payment[
        ["order_id"]
    ]
    .merge(
        order_item_summary,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

delivered_without_payment_details


,order_id,item_record_count,maximum_item_sequence,distinct_products,distinct_sellers,merchandise_value,freight_value,item_based_total
0,bfbd0f9bdef84302105ad712db648a6c,3,3,1,1,134.97,8.49,143.46


In [ ]:
financial_reconciliation = order_item_summary.merge(
    payment_summary,
    on="order_id",
    how="inner",
    validate="one_to_one"
)

financial_reconciliation["payment_item_difference"] = (
    financial_reconciliation["payment_total"]
    - financial_reconciliation["item_based_total"]
)

financial_reconciliation[
    "absolute_difference"
] = financial_reconciliation[
    "payment_item_difference"
].abs()

financial_reconciliation[
    [
        "payment_total",
        "item_based_total",
        "payment_item_difference",
        "absolute_difference"
    ]
].describe()

,payment_total,item_based_total,payment_item_difference,absolute_difference
count,"98,665.00","98,665.00","98,665.00","98,665.00"
mean,160.61,160.58,0.03,0.03
std,220.48,220.47,1.13,1.13
min,9.59,9.59,-51.62,0.00
25%,62.00,61.98,0.00,0.00
50%,105.29,105.29,0.00,0.00
75%,176.88,176.87,0.00,0.00
max,"13,664.08","13,664.08",182.81,182.81


In [ ]:
reconciliation_thresholds = pd.DataFrame({
    "threshold": [
        "Exact match",
        "Difference greater than 1 cent",
        "Difference greater than 1 real",
        "Difference greater than 10 reais"
    ],
    "number_of_orders": [
        (
            financial_reconciliation[
                "absolute_difference"
            ] == 0
        ).sum(),

        (
            financial_reconciliation[
                "absolute_difference"
            ] > 0.01
        ).sum(),

        (
            financial_reconciliation[
                "absolute_difference"
            ] > 1
        ).sum(),

        (
            financial_reconciliation[
                "absolute_difference"
            ] > 10
        ).sum()
    ]
})

reconciliation_thresholds

,threshold,number_of_orders
0,Exact match,78928
1,Difference greater than 1 cent,378
2,Difference greater than 1 real,249
3,Difference greater than 10 reais,98


### Financial-variable reconciliation

Order-level payment totals were compared with item prices plus freight for
98,665 orders present in both datasets. The median difference was zero and the
mean difference was R$0.03. Only 378 orders differed by more than R$0.01, 249
differed by more than R$1 and 98 differed by more than R$10. Many non-exact
matches below one cent are attributable to floating-point representation
rather than meaningful financial discrepancies.

The total recorded payment will be used as the principal monetary variable
because it represents the amount captured across all payment records,
including split-payment arrangements. Merchandise and freight values will be
retained separately. One delivered order has no payment record but contains
three item records with a combined item-and-freight value of R$143.46. For
this order only, the item-based total will be used as a documented fallback.

In [ ]:
financial_reconciliation_with_status = (
    financial_reconciliation
    .merge(
        data["orders"][["order_id", "order_status"]],
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

financial_difference_by_status = (
    financial_reconciliation_with_status
    .assign(
        difference_over_one_cent=
            financial_reconciliation_with_status[
                "absolute_difference"
            ] > 0.01,

        difference_over_one_real=
            financial_reconciliation_with_status[
                "absolute_difference"
            ] > 1,

        difference_over_ten_reais=
            financial_reconciliation_with_status[
                "absolute_difference"
            ] > 10
    )
    .groupby("order_status")
    .agg(
        number_of_orders=("order_id", "count"),
        mean_absolute_difference=(
            "absolute_difference", "mean"
        ),
        maximum_absolute_difference=(
            "absolute_difference", "max"
        ),
        differences_over_one_cent=(
            "difference_over_one_cent", "sum"
        ),
        differences_over_one_real=(
            "difference_over_one_real", "sum"
        ),
        differences_over_ten_reais=(
            "difference_over_ten_reais", "sum"
        )
    )
    .sort_values("number_of_orders", ascending=False)
    .reset_index()
)

financial_difference_by_status

,order_status,number_of_orders,mean_absolute_difference,maximum_absolute_difference,differences_over_one_cent,differences_over_one_real,differences_over_ten_reais
0,delivered,96477,0.03,182.81,372,246,97
1,shipped,1106,0.01,5.99,3,1,0
2,canceled,461,0.07,25.12,2,2,1
3,invoiced,312,0.00,0.01,1,0,0
4,processing,301,0.00,0.00,0,0,0
5,unavailable,6,0.00,0.00,0,0,0
6,approved,2,0.00,0.00,0,0,0


In [ ]:
largest_financial_differences = (
    financial_reconciliation_with_status
    .sort_values(
        "absolute_difference",
        ascending=False
    )
    [
        [
            "order_id",
            "order_status",
            "payment_total",
            "merchandise_value",
            "freight_value",
            "item_based_total",
            "payment_item_difference",
            "absolute_difference"
        ]
    ]
    .head(15)
)

largest_financial_differences

,order_id,order_status,payment_total,merchandise_value,freight_value,item_based_total,payment_item_difference,absolute_difference
79537,ce6d150fb29ada17d2082f4847107665,delivered,"1,586.47","1,299.00",104.66,"1,403.66",182.81,182.81
42515,6e5fe7366a2e1bfbf3257dba0af1267f,delivered,406.92,179.19,108.72,287.91,119.01,119.01
43434,70b742795bc441e94a44a084b6d9ce7a,delivered,578.82,269.99,196.94,466.93,111.89,111.89
58752,996c7e73600ad3723e8627ab7bef81e4,delivered,664.43,559.90,28.00,587.90,76.53,76.53
43437,70b7e94ea46d3e8b5bc12a50186edaf0,delivered,274.84,167.88,45.27,213.15,61.69,61.69
72496,bc2c82b0ef78d2252b6176d1972db7c9,delivered,303.02,165.00,77.01,242.01,61.01,61.01
67466,af9ffff2ce6b3defd34fd4c78857a379,delivered,466.97,395.65,17.52,413.17,53.80,53.80
14628,262118ce178bb3e4590a3adcf6d62e6b,delivered,126.12,119.80,57.94,177.74,-51.62,51.62
73899,bfdb5bbb06458d600a33d61f5f287472,delivered,394.36,297.00,51.93,348.93,45.43,45.43
54305,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,delivered,293.89,209.80,44.65,254.45,39.44,39.44


In [ ]:
order_analysis = (
    orders_with_customers
    .merge(
        order_item_summary,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        payment_summary,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        order_review_summary,
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

print(f"Original orders: {len(orders_with_customers):,}")
print(f"Analytical rows: {len(order_analysis):,}")
print(
    "Unique order IDs:",
    f"{order_analysis['order_id'].nunique():,}"
)

Original orders: 99,441
Analytical rows: 99,441
Unique order IDs: 99,441


In [ ]:
order_analysis["has_order_items"] = (
    order_analysis["item_record_count"]
    .notna()
    .astype(int)
)

order_analysis["has_payment"] = (
    order_analysis["payment_total"]
    .notna()
    .astype(int)
)

order_analysis["has_review"] = (
    order_analysis["final_review_score"]
    .notna()
    .astype(int)
)

In [ ]:
order_analysis["order_monetary_value"] = (
    order_analysis["payment_total"]
    .fillna(order_analysis["item_based_total"])
)

order_analysis["monetary_value_source"] = np.select(
    [
        order_analysis["payment_total"].notna(),
        (
            order_analysis["payment_total"].isna()
            & order_analysis["item_based_total"].notna()
        )
    ],
    [
        "payment_total",
        "item_total_fallback"
    ],
    default="missing"
)

monetary_source_summary = (
    order_analysis["monetary_value_source"]
    .value_counts(dropna=False)
    .rename_axis("monetary_value_source")
    .reset_index(name="number_of_orders")
)

monetary_source_summary

,monetary_value_source,number_of_orders
0,payment_total,99440
1,item_total_fallback,1


In [ ]:
review_zero_fill_columns = [
    "review_record_count",
    "distinct_review_ids",
    "any_review_title",
    "any_review_comment",
    "review_score_changed",
    "final_review_has_title",
    "final_review_has_comment"
]

order_analysis[review_zero_fill_columns] = (
    order_analysis[review_zero_fill_columns]
    .fillna(0)
)

order_analysis[review_zero_fill_columns] = (
    order_analysis[review_zero_fill_columns]
    .astype(int)
)

In [ ]:
order_analysis["purchase_year"] = (
    order_analysis["order_purchase_timestamp"].dt.year
)

order_analysis["purchase_month"] = (
    order_analysis["order_purchase_timestamp"].dt.month
)

order_analysis["purchase_year_month"] = (
    order_analysis["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

order_analysis["purchase_day_of_week"] = (
    order_analysis["order_purchase_timestamp"]
    .dt.day_name()
)

order_analysis["purchase_hour"] = (
    order_analysis["order_purchase_timestamp"].dt.hour
)

order_analysis[
    [
        "order_purchase_timestamp",
        "purchase_year",
        "purchase_month",
        "purchase_year_month",
        "purchase_day_of_week",
        "purchase_hour"
    ]
].head()

,order_purchase_timestamp,purchase_year,purchase_month,purchase_year_month,purchase_day_of_week,purchase_hour
0,2017-10-02 10:56:33,2017,10,2017-10,Monday,10
1,2018-07-24 20:41:37,2018,7,2018-07,Tuesday,20
2,2018-08-08 08:38:49,2018,8,2018-08,Wednesday,8
3,2017-11-18 19:28:06,2017,11,2017-11,Saturday,19
4,2018-02-13 21:18:39,2018,2,2018-02,Tuesday,21


In [ ]:
seconds_per_day = 24 * 60 * 60

order_analysis["approval_time_days"] = (
    order_analysis["order_approved_at"]
    - order_analysis["order_purchase_timestamp"]
).dt.total_seconds() / seconds_per_day

order_analysis["carrier_handover_days"] = (
    order_analysis["order_delivered_carrier_date"]
    - order_analysis["order_approved_at"]
).dt.total_seconds() / seconds_per_day

order_analysis["total_delivery_days"] = (
    order_analysis["order_delivered_customer_date"]
    - order_analysis["order_purchase_timestamp"]
).dt.total_seconds() / seconds_per_day

order_analysis["estimated_delivery_days"] = (
    order_analysis["order_estimated_delivery_date"]
    - order_analysis["order_purchase_timestamp"]
).dt.total_seconds() / seconds_per_day

order_analysis["delivery_delay_days"] = (
    order_analysis["order_delivered_customer_date"]
    - order_analysis["order_estimated_delivery_date"]
).dt.total_seconds() / seconds_per_day

In [ ]:
order_analysis["is_late_delivery"] = np.where(
    order_analysis["delivery_delay_days"].notna(),
    (
        order_analysis["delivery_delay_days"] > 0
    ).astype(int),
    np.nan
)

In [ ]:
duration_columns = [
    "approval_time_days",
    "carrier_handover_days",
    "total_delivery_days",
    "estimated_delivery_days",
    "delivery_delay_days"
]

duration_quality_records = []

for column in duration_columns:
    series = order_analysis[column]

    duration_quality_records.append({
        "feature": column,
        "non_missing_values": series.notna().sum(),
        "missing_values": series.isna().sum(),
        "negative_values": (series < 0).sum(),
        "zero_values": (series == 0).sum(),
        "minimum": series.min(),
        "median": series.median(),
        "maximum": series.max()
    })

duration_quality_summary = pd.DataFrame(
    duration_quality_records
)

duration_quality_summary

,feature,non_missing_values,missing_values,negative_values,zero_values,minimum,median,maximum
0,approval_time_days,99281,160,0,1296,0.00,0.01,187.88
1,carrier_handover_days,97644,1797,1359,0,-171.22,1.82,125.76
2,total_delivery_days,96476,2965,0,0,0.53,10.22,209.63
3,estimated_delivery_days,99441,0,0,0,1.65,23.24,155.14
4,delivery_delay_days,96476,2965,88649,0,-146.02,-11.95,188.98


In [ ]:
delivered_orders = (
    order_analysis[
        order_analysis["order_status"] == "delivered"
    ]
    .copy()
)

delivered_order_validation = pd.DataFrame({
    "metric": [
        "Delivered-order rows",
        "Unique delivered order IDs",
        "Delivered orders without items",
        "Delivered orders without monetary value",
        "Delivered orders without reviews",
        "Delivered orders without customer-delivery date"
    ],
    "value": [
        len(delivered_orders),
        delivered_orders["order_id"].nunique(),
        (
            delivered_orders["has_order_items"] == 0
        ).sum(),
        delivered_orders[
            "order_monetary_value"
        ].isna().sum(),
        (
            delivered_orders["has_review"] == 0
        ).sum(),
        delivered_orders[
            "order_delivered_customer_date"
        ].isna().sum()
    ]
})

delivered_order_validation

,metric,value
0,Delivered-order rows,96478
1,Unique delivered order IDs,96478
2,Delivered orders without items,0
3,Delivered orders without monetary value,0
4,Delivered orders without reviews,646
5,Delivered orders without customer-delivery date,8


### Monetary and operational-time findings

The completed order-level table contains 99,441 unique orders. Recorded
payment totals provide the monetary value for 99,440 orders, while the
documented item-based fallback supplies the value for the single order
without a payment record. All 96,478 delivered orders consequently have valid
item and monetary information.

Payment and item-based values reconcile closely. Among delivered orders, the
mean absolute difference is R$0.03, while only 97 orders differ by more than
R$10. Recorded payment is retained as the principal monetary measure because
it represents the amount recorded as paid by the customer. Item and freight
totals remain available as separate features and as a data-quality
comparison.

Duration analysis identified 1,359 records whose carrier timestamp precedes
the recorded approval timestamp. These values will be investigated and
flagged as operational timestamp inconsistencies rather than automatically
corrected.

The initial timestamp-based delivery-delay calculation produced no exact
zero-day delays because estimated delivery dates are recorded at midnight
while actual delivery timestamps include time of day. Delivery punctuality
will therefore be evaluated using normalised calendar dates. An order
delivered at any time on its estimated delivery date will be classified as
on time.

In [ ]:
actual_delivery_date = (
    order_analysis["order_delivered_customer_date"]
    .dt.normalize()
)

estimated_delivery_date = (
    order_analysis["order_estimated_delivery_date"]
    .dt.normalize()
)

order_analysis["delivery_delay_calendar_days"] = (
    actual_delivery_date
    - estimated_delivery_date
).dt.days

valid_delivery_dates = (
    actual_delivery_date.notna()
    & estimated_delivery_date.notna()
)

order_analysis["is_late_delivery"] = np.where(
    valid_delivery_dates,
    (
        actual_delivery_date
        > estimated_delivery_date
    ).astype(int),
    np.nan
)

In [ ]:
order_analysis["delivery_performance"] = np.select(
    [
        order_analysis[
            "delivery_delay_calendar_days"
        ].isna(),

        order_analysis[
            "delivery_delay_calendar_days"
        ] < 0,

        order_analysis[
            "delivery_delay_calendar_days"
        ] == 0,

        order_analysis[
            "delivery_delay_calendar_days"
        ] > 0
    ],
    [
        "delivery_date_missing",
        "early",
        "on_estimated_date",
        "late"
    ],
    default="unclassified"
)

In [ ]:
delivery_performance_validation = (
    order_analysis[
        order_analysis["order_status"] == "delivered"
    ]
    ["delivery_performance"]
    .value_counts(dropna=False)
    .rename_axis("delivery_performance")
    .reset_index(name="number_of_orders")
)

delivery_performance_validation[
    "percentage"
] = (
    delivery_performance_validation[
        "number_of_orders"
    ]
    / delivery_performance_validation[
        "number_of_orders"
    ].sum()
    * 100
).round(2)

delivery_performance_validation

,delivery_performance,number_of_orders,percentage
0,early,88644,91.88
1,late,6534,6.77
2,on_estimated_date,1292,1.34
3,delivery_date_missing,8,0.01


In [ ]:
negative_carrier_handover_by_status = (
    order_analysis
    .assign(
        negative_carrier_handover=(
            order_analysis["carrier_handover_days"] < 0
        )
    )
    .groupby("order_status")
    .agg(
        orders_with_carrier_duration=(
            "carrier_handover_days", "count"
        ),
        negative_carrier_durations=(
            "negative_carrier_handover", "sum"
        )
    )
    .reset_index()
)

negative_carrier_handover_by_status[
    "negative_percentage"
] = (
    negative_carrier_handover_by_status[
        "negative_carrier_durations"
    ]
    / negative_carrier_handover_by_status[
        "orders_with_carrier_duration"
    ]
    * 100
).round(2)

negative_carrier_handover_by_status = (
    negative_carrier_handover_by_status
    .sort_values(
        "negative_carrier_durations",
        ascending=False
    )
    .reset_index(drop=True)
)

negative_carrier_handover_by_status


,order_status,orders_with_carrier_duration,negative_carrier_durations,negative_percentage
0,delivered,96462,1350,1.40
1,shipped,1107,9,0.81
2,canceled,75,0,0.00
3,approved,0,0,NaN
4,created,0,0,NaN
5,invoiced,0,0,NaN
6,processing,0,0,NaN
7,unavailable,0,0,NaN


In [ ]:
negative_carrier_cases = (
    order_analysis[
        order_analysis["carrier_handover_days"] < 0
    ]
    .copy()
)

negative_carrier_magnitude = (
    negative_carrier_cases[
        "carrier_handover_days"
    ]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
    .to_frame(name="carrier_handover_days")
)

negative_carrier_magnitude

,carrier_handover_days
count,"1,359.00"
mean,-1.03
std,4.80
min,-171.22
1%,-6.89
5%,-3.62
25%,-1.08
50%,-0.72
75%,-0.06
95%,-0.01


In [ ]:
extreme_negative_carrier_cases = (
    negative_carrier_cases
    .sort_values("carrier_handover_days")
    [
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "approval_time_days",
            "carrier_handover_days",
            "total_delivery_days"
        ]
    ]
    .head(15)
)

extreme_negative_carrier_cases

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,approval_time_days,carrier_handover_days,total_delivery_days
25883,7c48bb55e8e4f7e56d412e9653db37bc,delivered,2018-07-16 18:40:53,2018-07-16 18:50:22,2018-01-26 13:35:00,2018-07-23 20:04:45,0.01,-171.22,7.06
14562,1fab4ac9d85079b3da72a11475ae1685,delivered,2017-09-01 19:04:22,2017-09-13 22:06:11,2017-09-04 13:10:23,2017-09-08 20:13:03,12.13,-9.37,7.05
46163,0184d4ddb259e1a4cfc2871888cf97b8,delivered,2017-09-01 20:04:28,2017-09-13 22:17:15,2017-09-04 14:05:50,2017-09-09 15:12:44,12.09,-9.34,7.80
98710,1378f9601350615613cc8832d6789c5d,delivered,2017-09-01 20:28:02,2017-09-13 22:03:51,2017-09-04 18:07:55,2017-09-13 22:24:46,12.07,-9.16,12.08
41592,8554cb37f7158cb0b082a841d24a4589,delivered,2017-09-01 18:40:44,2017-09-13 21:58:04,2017-09-04 19:12:19,2017-09-08 20:07:45,12.14,-9.12,7.06
11738,cf72398d0690f841271b695bbfda82d2,delivered,2017-09-01 18:45:33,2017-09-13 22:04:39,2017-09-04 20:12:41,2017-09-11 14:15:02,12.14,-9.08,9.81
85393,40de47dfa620d667117e4a6067b6e1ec,delivered,2017-09-01 20:05:55,2017-09-13 21:58:38,2017-09-04 20:36:58,2017-09-08 20:41:16,12.08,-9.06,7.02
31211,bc4854efd86d9f42140c951c595d20c1,delivered,2017-09-01 20:05:42,2017-09-13 22:00:51,2017-09-04 20:49:57,2017-09-06 22:15:45,12.08,-9.05,5.09
55302,77ca435b03fbf991e5027e3776e37885,delivered,2017-09-01 18:49:54,2017-09-13 21:56:08,2017-09-04 20:50:00,2017-09-30 17:33:43,12.13,-9.05,28.95
68315,580f3268bc2075c5961b2d2929c7a35b,delivered,2017-09-01 20:17:09,2017-09-13 22:03:42,2017-09-05 11:58:18,2017-09-18 17:52:20,12.07,-8.42,16.90


In [ ]:
order_analysis["purchase_to_carrier_days"] = (
    order_analysis["order_delivered_carrier_date"]
    - order_analysis["order_purchase_timestamp"]
).dt.total_seconds() / seconds_per_day

purchase_to_carrier_quality = pd.DataFrame({
    "metric": [
        "Non-missing purchase-to-carrier values",
        "Missing purchase-to-carrier values",
        "Negative purchase-to-carrier values",
        "Zero purchase-to-carrier values",
        "Minimum days",
        "Median days",
        "Maximum days"
    ],
    "value": [
        order_analysis[
            "purchase_to_carrier_days"
        ].notna().sum(),

        order_analysis[
            "purchase_to_carrier_days"
        ].isna().sum(),

        (
            order_analysis[
                "purchase_to_carrier_days"
            ] < 0
        ).sum(),

        (
            order_analysis[
                "purchase_to_carrier_days"
            ] == 0
        ).sum(),

        order_analysis[
            "purchase_to_carrier_days"
        ].min(),

        order_analysis[
            "purchase_to_carrier_days"
        ].median(),

        order_analysis[
            "purchase_to_carrier_days"
        ].max()
    ]
})

purchase_to_carrier_quality

,metric,value
0,Non-missing purchase-to-carrier values,"97,658.00"
1,Missing purchase-to-carrier values,"1,783.00"
2,Negative purchase-to-carrier values,166.00
3,Zero purchase-to-carrier values,0.00
4,Minimum days,-171.21
5,Median days,2.20
6,Maximum days,125.78


In [ ]:
order_analysis["carrier_timing_inconsistent"] = (
    order_analysis["carrier_handover_days"] < 0
).astype(int)

order_analysis["carrier_handover_days_clean"] = (
    order_analysis["carrier_handover_days"]
    .where(
        order_analysis["carrier_handover_days"] >= 0
    )
)

In [ ]:
delivered_orders = (
    order_analysis[
        order_analysis["order_status"] == "delivered"
    ]
    .copy()
)

print(f"Delivered orders: {len(delivered_orders):,}")
print(
    "Unique delivered order IDs:",
    f"{delivered_orders['order_id'].nunique():,}"
)

Delivered orders: 96,478
Unique delivered order IDs: 96,478


In [ ]:
valid_delivered_dates = delivered_orders[
    delivered_orders[
        "delivery_delay_calendar_days"
    ].notna()
].copy()

delivery_kpi_summary = pd.DataFrame({
    "metric": [
        "Delivered orders",
        "Orders with usable delivery dates",
        "Orders delivered early",
        "Orders delivered on estimated date",
        "Orders delivered late",
        "Late-delivery percentage",
        "Median total delivery days",
        "Median calendar days before estimate",
        "95th percentile total delivery days",
        "Maximum total delivery days"
    ],
    "value": [
        len(delivered_orders),

        len(valid_delivered_dates),

        (
            valid_delivered_dates[
                "delivery_performance"
            ] == "early"
        ).sum(),

        (
            valid_delivered_dates[
                "delivery_performance"
            ] == "on_estimated_date"
        ).sum(),

        (
            valid_delivered_dates[
                "delivery_performance"
            ] == "late"
        ).sum(),

        round(
            valid_delivered_dates[
                "is_late_delivery"
            ].mean() * 100,
            2
        ),

        round(
            valid_delivered_dates[
                "total_delivery_days"
            ].median(),
            2
        ),

        round(
            -valid_delivered_dates[
                "delivery_delay_calendar_days"
            ].median(),
            2
        ),

        round(
            valid_delivered_dates[
                "total_delivery_days"
            ].quantile(0.95),
            2
        ),

        round(
            valid_delivered_dates[
                "total_delivery_days"
            ].max(),
            2
        )
    ]
})

delivery_kpi_summary

,metric,value
0,Delivered orders,"96,478.00"
1,Orders with usable delivery dates,"96,470.00"
2,Orders delivered early,"88,644.00"
3,Orders delivered on estimated date,"1,292.00"
4,Orders delivered late,"6,534.00"
5,Late-delivery percentage,6.77
6,Median total delivery days,10.22
7,Median calendar days before estimate,12.00
8,95th percentile total delivery days,29.27
9,Maximum total delivery days,209.63
